# DATA11010 ML1, Exercise Set 3, Problem 12: MLP and CNN for time series classification

## Instructions

* The problem statement is Problem 12 of the Exercise Set 3 PDF. Write your answers, with the figures and tables they
  need, in your report, not in this notebook.
* Write the code of Task a: the functions `MLP` and `CNN`, each returning a model built with `nn.Sequential`. Small
  tests check the output shapes. The rest of the code is given: then run the notebook from top to bottom.
* The notebook runs in Google Colab or locally (packages: `numpy`, `pandas`, `matplotlib`, `torch`); a CPU is enough.
  On a standard Colab CPU runtime the whole notebook takes about 3 minutes, most of it in Task b2 (about 2 minutes).
* Data: the notebook reads `TwoPatterns_TRAIN.txt` and `TwoPatterns_TEST.txt` from the folder of the notebook or
  downloads them if they are not there.

## Data: TwoPatterns

A simulated data set (P. Geurts, 2002) from the UCR time series archive
([Dau et al., 2019](https://doi.org/10.1109/JAS.2019.1911747)).

* Each series has 128 time steps: noise with two embedded patterns at random positions. An **up** pattern steps from
  a low to a high level, a **down** pattern from a high to a low level.
* The class is the order of the two patterns: 1 down-down, 2 up-down, 3 down-up, 4 up-up (in the notebook the labels
  are 0 to 3).
* 1000 training and 4000 test series; each series is standardised to mean 0 and standard deviation 1.
* File format: one series per line; the first column is the class, the remaining 128 columns are the values.

In [ ]:
import io, urllib.request, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
# Read the data files from the folder of the notebook; download them if they are missing.
DATA_URL = "https://www.timeseriesclassification.com/aeon-toolkit/TwoPatterns.zip"
files = {part: Path(f"TwoPatterns_{part}.txt") for part in ("TRAIN", "TEST")}
if not all(f.exists() for f in files.values()):
    print("Data files not found in", Path.cwd().resolve(), "- downloading", DATA_URL)
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(DATA_URL).read())) as z:
        for f in files.values():
            f.write_bytes(z.read(f.name))
train_data, test_data = (np.loadtxt(files[part]) for part in ("TRAIN", "TEST"))

def to_tensors(a):  # first column: class 1..4, the rest: the series
    return torch.tensor(a[:, 1:], dtype=torch.float32), torch.tensor(a[:, 0].astype(int) - 1)

X_train, y_train = to_tensors(train_data)
X_test, y_test = to_tensors(test_data)
print("training set:", tuple(X_train.shape), " test set:", tuple(X_test.shape))

class_names = ["down-down", "up-down", "down-up", "up-up"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for k, ax in enumerate(axes):
    for series in X_train[y_train == k][:3]:
        ax.plot(series)
    ax.set(title=f"class {k}: {class_names[k]}", xlabel="time")
plt.show()

## Task a: models and number of parameters

Write the two models:
* **MLP:** input 128 values, two hidden layers with 64 and 32 neurons and ReLU activations,
  output layer with 4 neurons (one logit per class).
* **CNN:** two blocks of [1D convolution with 16 channels and kernel size 7, ReLU, max pooling with window 4],
  then a fully connected layer from the flattened result to the 4 outputs.

`MLP` and `CNN` are functions that return a new model, not fixed models, because each training run of Task b needs a
fresh, newly initialised model.

In [ ]:
def MLP(n_inputs=128, hidden_layer_sizes=(64, 32), n_outputs=4):
    # ======= SOLUTION (hint: nn.Sequential of nn.Linear and nn.ReLU layers, no ReLU after the last nn.Linear) =======
    model = ...  # nn.Sequential(...)
    return model
    # ======= END SOLUTION =======

In [ ]:
def test_mlp(batch_size=5):
    x = torch.randn(batch_size, 128)  # a batch of series in the format of X_train
    out = MLP()(x)
    assert out.shape == (batch_size, 4), f"MLP output shape {tuple(out.shape)}, expected {(batch_size, 4)}"
    print("MLP: output shape is correct, the test is passed.")

test_mlp()

In [ ]:
def CNN(length=128, n_outputs=4, channels=16, kernel_size=7, pool=4):
    # n is the length of the series after the two convolution + pooling blocks, so the last nn.Linear has
    # n * channels inputs.
    n = ((length - kernel_size + 1) // pool - kernel_size + 1) // pool
    # ======= SOLUTION (hint: nn.Sequential that starts with nn.Unflatten(1, (1, length)), because nn.Conv1d expects
    # input of shape (batch, channels, length); then nn.Conv1d, nn.ReLU and nn.MaxPool1d twice; then nn.Flatten and
    # nn.Linear) =======
    model = ...  # nn.Sequential(...)
    return model
    # ======= END SOLUTION =======

In [ ]:
def test_cnn(batch_size=5):
    x = torch.randn(batch_size, 128)  # a batch of series in the format of X_train
    out = CNN()(x)
    assert out.shape == (batch_size, 4), f"CNN output shape {tuple(out.shape)}, expected {(batch_size, 4)}"
    print("CNN: output shape is correct, the test is passed.")

test_cnn()

In [ ]:
# Given: the number of trainable parameters (weights and biases) of a model. Compare the counts printed below
# with your hand calculation.
def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for model in (MLP(), CNN()):
    print(model)
    for name, p in model.named_parameters():
        print(f"  {name:20s} {str(tuple(p.shape)):15s} {p.numel():6d}")
    print("  total:", n_params(model), "\n")

## Task b: training and comparison

`train` trains a model with Adam and the cross-entropy loss on the first `n_train` training series (the training
series are in random order) and returns the training and test errors (fraction misclassified) and the losses after
each epoch.

In [ ]:
def error(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) != y).float().mean().item()

def train(model, n_train, n_epochs=100, batch_size=32, lr=0.001):
    X, y = X_train[:n_train], y_train[:n_train]
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    train_losses, test_losses = [], []
    for epoch in range(n_epochs):
        perm = torch.randperm(n_train)
        for i in range(0, n_train, batch_size):
            batch = perm[i:i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(X[batch]), y[batch])
            loss.backward()
            optimizer.step()
        with torch.no_grad():  # losses after each epoch
            train_losses.append(loss_fn(model(X), y).item())
            test_losses.append(loss_fn(model(X_test), y_test).item())
    return error(model, X, y), error(model, X_test, y_test), train_losses, test_losses

def plot_losses(losses, names, title=""):
    for l, name in zip(losses, names):
        plt.plot(range(1, len(l) + 1), l, label=name)
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("cross-entropy loss")
    plt.title(title)
    plt.legend()
    plt.show()

### b1. One run of each model, 100 training series

In [ ]:
n_train = 100
runs = {}
for name, make_model in [("MLP", MLP), ("CNN", CNN)]:
    torch.manual_seed(0)
    runs[name] = train(make_model(), n_train)
    print(f"{name}: training error {runs[name][0]:.3f}, test error {runs[name][1]:.3f}")
plot_losses([runs[m][i] for m in runs for i in (2, 3)],
            [f"{m} {part}" for m in runs for part in ("train", "test")], f"n_train = {n_train}, seed 0")

### b2. Training set size and three seeds

Each model is trained with 100, 300 and 1000 training series and three seeds. The test losses of the runs with 100
training series are kept for the plot of the three seeds below. This cell takes most of the run time of the notebook,
about 2 minutes on a standard Colab CPU runtime.

In [ ]:
results, seed_losses = [], {}
for n_train in [100, 300, 1000]:
    for seed in range(3):
        for name, make_model in [("MLP", MLP), ("CNN", CNN)]:
            torch.manual_seed(seed)
            train_error, test_error, _, test_losses = train(make_model(), n_train)
            results.append({"n_train": n_train, "model": name, "seed": seed,
                            "training error": train_error, "test error": test_error})
            if n_train == 100:
                seed_losses[name, seed] = test_losses
results = pd.DataFrame(results)
results.groupby(["n_train", "model"])[["training error", "test error"]].agg(["mean", "std"]).round(3)

In [ ]:
# The three seeds with 100 training series (from the runs above; nothing is trained again)
keys = [(name, seed) for name in ("MLP", "CNN") for seed in range(3)]
plot_losses([seed_losses[k] for k in keys], [f"{name} test, seed {seed}" for name, seed in keys], "n_train = 100")
results[results["n_train"] == 100].pivot(index="seed", columns="model", values="test error").round(3)